# Knudsen number from a hard-sphere mean free path

## Prerequisites

Review ideal-gas number density, hard-sphere mean free path, and dimensionless scale analysis.

## Objectives

1. Calculate pressure-dependent mean free paths for an illustrative hard-sphere ideal gas.
2. Evaluate the unitless Knudsen number $\mathrm{Kn}=\lambda/L$ with an explicit characteristic length.
3. Verify the expected inverse-pressure scaling at fixed temperature and collision diameter.
4. Plot $\mathrm{Kn}$ without embedding disputed regime-classification thresholds.

All gas parameters are synthetic illustrative inputs rather than fitted material data.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.thermal.statmech.kinetic_theory import (
    HardSphereIdealGasState,
)
from projectkoios.physkit.thermal.statmech.transport.knudsen import (
    KnudsenNumberModel,
)
from projectkoios.physkit.units import PhysicalUnit, ScalarQuantity, VectorQuantity

## Definition

For mean free path $\lambda$ and characteristic system length $L$,

$$
\mathrm{Kn}=\frac{\lambda}{L}.
$$

The value compares a molecular collision length scale with a geometric scale. Any mapping from $\mathrm{Kn}$ to named flow regimes requires a separately specified convention and is intentionally outside this model.


In [ ]:
minimum_pressure_pascal = 1.0e-3
maximum_pressure_pascal = 1.0e5
pressure_sample_count = 400
temperature = ScalarQuantity(
    magnitude=300.0,
    unit=PhysicalUnit(expression="kelvin"),
)
collision_diameter = ScalarQuantity(
    magnitude=0.37,
    unit=PhysicalUnit(expression="nanometer"),
)
characteristic_length = ScalarQuantity(
    magnitude=1.0,
    unit=PhysicalUnit(expression="millimeter"),
)
pressure_values_pascal = np.geomspace(
    minimum_pressure_pascal,
    maximum_pressure_pascal,
    pressure_sample_count,
).astype(np.float64)
mean_free_path_values_metre = np.array(
    [
        HardSphereIdealGasState(
            pressure=ScalarQuantity(
                magnitude=float(pressure_pascal),
                unit=PhysicalUnit(expression="pascal"),
            ),
            temperature=temperature,
            collision_diameter=collision_diameter,
        ).mean_free_path.magnitude
        for pressure_pascal in pressure_values_pascal
    ],
    dtype=np.float64,
)
mean_free_paths = VectorQuantity(
    magnitude=mean_free_path_values_metre,
    unit=PhysicalUnit(expression="meter"),
)
knudsen_model = KnudsenNumberModel(characteristic_length=characteristic_length)
knudsen_evaluation = knudsen_model.evaluate(mean_free_paths=mean_free_paths)

In [ ]:
logarithmic_slope = np.polyfit(
    np.log(pressure_values_pascal),
    np.log(knudsen_evaluation.knudsen_numbers.magnitude),
    deg=1,
)[0]
scaling_absolute_tolerance = 2.0e-15
assert np.isclose(
    logarithmic_slope,
    -1.0,
    rtol=0.0,
    atol=scaling_absolute_tolerance,
)
pressure_scaled_knudsen = (
    pressure_values_pascal * knudsen_evaluation.knudsen_numbers.magnitude
)
assert np.allclose(
    pressure_scaled_knudsen,
    pressure_scaled_knudsen[0],
    rtol=5.0e-16,
    atol=0.0,
)

In [ ]:
figure_size_inches = (8.0, 4.8)
figure, axis = plt.subplots(figsize=figure_size_inches)
axis.loglog(
    pressure_values_pascal,
    knudsen_evaluation.knudsen_numbers.magnitude,
    label="hard-sphere ideal-gas Knudsen number",
)
equal_length_scale = 1.0
axis.axhline(
    equal_length_scale,
    color="tab:red",
    linestyle="--",
    label=r"$\lambda=L$",
)
axis.set(
    xlabel="pressure (Pa)",
    ylabel="Knudsen number",
    title="Illustrative Knudsen number for a 1 mm length scale",
)
axis.grid(True, which="both")
axis.legend()
figure.tight_layout()
plt.show()

## Interpretation

At fixed temperature and collision diameter, the hard-sphere ideal-gas mean free path is proportional to $P^{-1}$. Because the characteristic length is fixed, the Knudsen number has the same inverse-pressure scaling. The horizontal line at $\mathrm{Kn}=1$ marks equality of the two represented length scales; it is not a complete flow-regime classifier.

The plot verifies the documented idealized model and unit conversions. It does not validate a real gas, device geometry, or flow-regime boundary.

## Exercises

1. Change the characteristic length and predict the vertical shift on the logarithmic plot.
2. Derive the dependence of $\mathrm{Kn}$ on collision diameter and temperature.
3. Replace the illustrative gas inputs with verified reference data and record their provenance before making material claims.
4. Specify and test a separate regime convention if named flow classifications are required.
